In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.feature_selection import (
    SelectKBest, f_regression, mutual_info_regression,
    RFE, SequentialFeatureSelector
)
import pickle
import joblib
import os
import psutil
import threading
from concurrent.futures import ThreadPoolExecutor
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================================================
# STEP 1: LOAD DATA AND TRAIN BASE MODEL
# ============================================================================

print("\n" + "="*80)
print("STEP 1: TRAINING BASE MODEL")
print("="*80)

# Load data
data = load_diabetes()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

# Prepare data
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train base model
base_model = LinearRegression()
base_model.fit(X_train, y_train)

# Evaluate base model
y_pred_base = base_model.predict(X_test)
base_mse = mean_squared_error(y_test, y_pred_base)
base_r2 = r2_score(y_test, y_pred_base)

print(f"✓ Base model trained")
print(f"  MSE: {base_mse:.2f}")
print(f"  R²: {base_r2:.4f}")

# Save base model
joblib.dump(base_model, 'base_model.pkl')
base_model_size = os.path.getsize('base_model.pkl') / 1024

print(f"  Model size: {base_model_size:.2f} KB")


STEP 1: TRAINING BASE MODEL
✓ Base model trained
  MSE: 2900.19
  R²: 0.4526
  Model size: 1.07 KB


In [3]:
# ============================================================================
# STEP 2: MEASURE BASE MODEL RESPONSE TIME
# ============================================================================

print("\n" + "="*80)
print("STEP 2: MEASURING BASE MODEL RESPONSE TIME")
print("="*80)

def measure_response_time(model, X_data, num_predictions=1000):
    """Measure average response time."""
    times = []
    
    for i in range(num_predictions):
        sample = X_data.iloc[i % len(X_data):i % len(X_data) + 1]
        
        start = time.perf_counter()
        prediction = model.predict(sample)
        end = time.perf_counter()
        
        times.append((end - start) * 1000)  # ms
    
    return {
        'avg_time_ms': np.mean(times),
        'min_time_ms': np.min(times),
        'max_time_ms': np.max(times),
        'p95_time_ms': np.percentile(times, 95),
        'p99_time_ms': np.percentile(times, 99),
        'throughput_per_sec': 1000 / np.mean(times),
        'times': times
    }

print("Measuring base model response time...")
base_response = measure_response_time(base_model, X_test)

print(f"\nBase Model Response Time:")
print(f"  Average: {base_response['avg_time_ms']:.4f} ms")
print(f"  P95: {base_response['p95_time_ms']:.4f} ms")
print(f"  P99: {base_response['p99_time_ms']:.4f} ms")
print(f"  Throughput: {base_response['throughput_per_sec']:.2f} predictions/second")


STEP 2: MEASURING BASE MODEL RESPONSE TIME
Measuring base model response time...

Base Model Response Time:
  Average: 0.7373 ms
  P95: 0.8009 ms
  P99: 0.8660 ms
  Throughput: 1356.29 predictions/second


In [4]:
# ============================================================================
# STEP 3: PRUNING - REMOVE LESS IMPORTANT FEATURES
# ============================================================================

print("\n" + "="*80)
print("STEP 3: MODEL PRUNING")
print("="*80)

print("""
PRUNING STRATEGY FOR LINEAR REGRESSION:
- Linear regression doesn't have "weights" to prune like neural networks
- Instead, we perform FEATURE PRUNING (feature selection)
- Remove features with low importance/coefficients
- This reduces model complexity and inference time
""")

# Get feature importance from coefficients
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coefficient': base_model.coef_,
    'abs_coefficient': np.abs(base_model.coef_)
}).sort_values('abs_coefficient', ascending=False)

print("\nFeature Importance (by coefficient magnitude):")
print(feature_importance.to_string(index=False))

# Pruning strategy: Keep top N most important features
def prune_model(X_train, X_test, y_train, y_test, top_n_features=5):
    """
    Prune model by selecting top N most important features.
    
    Parameters:
    -----------
    top_n_features : int
        Number of features to keep
    
    Returns:
    --------
    pruned_model : trained model with fewer features
    selected_features : list of selected feature names
    """
    # Train temporary model to get coefficients
    temp_model = LinearRegression()
    temp_model.fit(X_train, y_train)
    
    # Get feature importance
    feature_imp = pd.DataFrame({
        'feature': X_train.columns,
        'abs_coef': np.abs(temp_model.coef_)
    }).sort_values('abs_coef', ascending=False)
    
    # Select top N features
    selected_features = feature_imp.head(top_n_features)['feature'].tolist()
    
    # Train new model with selected features only
    X_train_pruned = X_train[selected_features]
    X_test_pruned = X_test[selected_features]
    
    pruned_model = LinearRegression()
    pruned_model.fit(X_train_pruned, y_train)
    
    return pruned_model, selected_features, X_train_pruned, X_test_pruned

# Test different pruning levels
pruning_levels = [8, 6, 5, 4, 3]
pruning_results = []

print("\nTesting different pruning levels:")
print("-"*80)

for n_features in pruning_levels:
    pruned_model, selected_features, X_train_pruned, X_test_pruned = prune_model(
        X_train, X_test, y_train, y_test, top_n_features=n_features
    )
    
    # Evaluate pruned model
    y_pred_pruned = pruned_model.predict(X_test_pruned)
    pruned_mse = mean_squared_error(y_test, y_pred_pruned)
    pruned_r2 = r2_score(y_test, y_pred_pruned)
    
    # Save pruned model
    model_file = f'pruned_model_{n_features}features.pkl'
    joblib.dump({
        'model': pruned_model,
        'selected_features': selected_features
    }, model_file)
    
    model_size = os.path.getsize(model_file) / 1024
    
    # Measure response time
    response = measure_response_time(pruned_model, X_test_pruned, num_predictions=500)
    
    result = {
        'n_features': n_features,
        'selected_features': selected_features,
        'mse': pruned_mse,
        'r2': pruned_r2,
        'model_size_kb': model_size,
        'avg_response_ms': response['avg_time_ms'],
        'p95_response_ms': response['p95_time_ms'],
        'throughput': response['throughput_per_sec'],
        'model': pruned_model,
        'X_test_pruned': X_test_pruned
    }
    
    pruning_results.append(result)
    
    print(f"\n{n_features} Features: {selected_features}")
    print(f"  MSE: {pruned_mse:.2f} (Δ: {pruned_mse - base_mse:+.2f})")
    print(f"  R²: {pruned_r2:.4f} (Δ: {pruned_r2 - base_r2:+.4f})")
    print(f"  Model size: {model_size:.2f} KB ({(base_model_size - model_size)/base_model_size*100:.1f}% smaller)")
    print(f"  Avg response: {response['avg_time_ms']:.4f} ms ({(base_response['avg_time_ms'] - response['avg_time_ms'])/base_response['avg_time_ms']*100:+.1f}%)")

# Select best pruned model (balance between accuracy and efficiency)
print("\n" + "="*80)
print("SELECTING BEST PRUNED MODEL")
print("="*80)

# Find model with best balance (R² > 0.4 and smallest size)
valid_models = [r for r in pruning_results if r['r2'] > 0.4]

if valid_models:
    best_pruned = min(valid_models, key=lambda x: x['n_features'])
    print(f"\n✓ Best pruned model: {best_pruned['n_features']} features")
    print(f"  Features: {best_pruned['selected_features']}")
    print(f"  R²: {best_pruned['r2']:.4f}")
    print(f"  Size reduction: {(base_model_size - best_pruned['model_size_kb'])/base_model_size*100:.1f}%")
    print(f"  Speed improvement: {(base_response['avg_time_ms'] - best_pruned['avg_response_ms'])/base_response['avg_time_ms']*100:.1f}%")
else:
    best_pruned = pruning_results[0]
    print(f"\n⚠️ Using least pruned model: {best_pruned['n_features']} features")


STEP 3: MODEL PRUNING

PRUNING STRATEGY FOR LINEAR REGRESSION:
- Linear regression doesn't have "weights" to prune like neural networks
- Instead, we perform FEATURE PRUNING (feature selection)
- Remove features with low importance/coefficients
- This reduces model complexity and inference time


Feature Importance (by coefficient magnitude):
feature  coefficient  abs_coefficient
     s1  -931.488846       931.488846
     s5   736.198859       736.198859
    bmi   542.428759       542.428759
     s2   518.062277       518.062277
     bp   347.703844       347.703844
     s4   275.317902       275.317902
    sex  -241.964362       241.964362
     s3   163.419983       163.419983
     s6    48.670657        48.670657
    age    37.904021        37.904021

Testing different pruning levels:
--------------------------------------------------------------------------------

8 Features: ['s1', 's5', 'bmi', 's2', 'bp', 's4', 'sex', 's3']
  MSE: 2882.72 (Δ: -17.47)
  R²: 0.4559 (Δ: +0.0033)
  M

In [5]:
# ============================================================================
# STEP 4: QUANTIZATION - REDUCE COEFFICIENT PRECISION
# ============================================================================

print("\n" + "="*80)
print("STEP 4: MODEL QUANTIZATION")
print("="*80)

print("""
QUANTIZATION STRATEGY FOR LINEAR REGRESSION:
- Reduce coefficient precision (FP64 → FP32 → FP16 → INT8)
- Store coefficients with lower precision
- Trade accuracy for smaller model size
- Faster computations on specialized hardware
""")

class QuantizedLinearRegression:
    """Quantized Linear Regression with reduced precision."""
    
    def __init__(self, base_model, dtype=np.float32):
        """
        Quantize model coefficients.
        
        Parameters:
        -----------
        base_model : LinearRegression
        dtype : numpy dtype (np.float32, np.float16, or custom INT8)
        """
        self.dtype = dtype
        self.dtype_name = str(dtype).split("'")[1] if hasattr(dtype, '__name__') else 'INT8'
        
        # Store feature names
        self.feature_names_ = None
        
        # Quantize coefficients
        if dtype == 'int8':
            # INT8 quantization
            self.coef_min = base_model.coef_.min()
            self.coef_max = base_model.coef_.max()
            self.coef_scale = (self.coef_max - self.coef_min) / 255
            self.coef_zero_point = -128
            
            self.coef_ = np.round(
                (base_model.coef_ - self.coef_min) / self.coef_scale + self.coef_zero_point
            ).astype(np.int8)
            
            self.intercept_ = base_model.intercept_
            self.is_int8 = True
        else:
            # Float quantization (FP32, FP16)
            self.coef_ = base_model.coef_.astype(dtype)
            self.intercept_ = base_model.intercept_.astype(dtype)
            self.is_int8 = False
    
    def predict(self, X):
        """Make predictions."""
        X_array = X.values if hasattr(X, 'values') else X
        
        if self.is_int8:
            # Dequantize INT8 coefficients
            coef_dequantized = (self.coef_.astype(np.float64) - self.coef_zero_point) * self.coef_scale + self.coef_min
            predictions = X_array @ coef_dequantized + self.intercept_
        else:
            X_quantized = X_array.astype(self.dtype)
            predictions = X_quantized @ self.coef_ + self.intercept_
        
        return predictions.astype(np.float64)
    
    def save(self, filename):
        """Save quantized model."""
        if self.is_int8:
            data = {
                'coef': self.coef_,
                'coef_min': self.coef_min,
                'coef_max': self.coef_max,
                'coef_scale': self.coef_scale,
                'coef_zero_point': self.coef_zero_point,
                'intercept': self.intercept_,
                'is_int8': True,
                'feature_names': self.feature_names_
            }
        else:
            data = {
                'coef': self.coef_,
                'intercept': self.intercept_,
                'dtype': self.dtype_name,
                'is_int8': False,
                'feature_names': self.feature_names_
            }
        np.savez_compressed(filename, **data)
        
    @classmethod
    def load(cls, filename):
        """Load quantized model."""
        data = np.load(filename, allow_pickle=True)
        model = cls.__new__(cls)
        
        model.coef_ = data['coef']
        model.intercept_ = data['intercept']
        model.is_int8 = data['is_int8']
        model.feature_names_ = data['feature_names'].tolist() if 'feature_names' in data else None
        
        if model.is_int8:
            model.coef_min = data['coef_min']
            model.coef_max = data['coef_max']
            model.coef_scale = data['coef_scale']
            model.coef_zero_point = data['coef_zero_point']
            model.dtype_name = 'INT8'
        else:
            model.dtype_name = str(data['dtype'])
        
        return model

# Apply INT8 quantization
quantized_model = QuantizedLinearRegression(base_model, dtype='int8')
quantized_model.feature_names_ = list(X.columns)

# Evaluate quantized model
y_pred_quantized = quantized_model.predict(X_test)
quantized_mse = mean_squared_error(y_test, y_pred_quantized)
quantized_r2 = r2_score(y_test, y_pred_quantized)

# Save quantized model
quantized_file = 'quantized_model_int8.npz'
quantized_model.save(quantized_file)
quantized_size = os.path.getsize(quantized_file) / 1024

print(f"✓ Model quantized to INT8")
print(f"  MSE: {quantized_mse:.2f} (Δ: {quantized_mse - base_mse:+.2f})")
print(f"  R²: {quantized_r2:.4f} (Δ: {quantized_r2 - base_r2:+.4f})")
print(f"  Size: {quantized_size:.2f} KB ({(base_model_size - quantized_size)/base_model_size*100:.1f}% smaller)")
print(f"  Compression: {base_model_size / quantized_size:.2f}x")


STEP 4: MODEL QUANTIZATION

QUANTIZATION STRATEGY FOR LINEAR REGRESSION:
- Reduce coefficient precision (FP64 → FP32 → FP16 → INT8)
- Store coefficients with lower precision
- Trade accuracy for smaller model size
- Faster computations on specialized hardware

✓ Model quantized to INT8
  MSE: 2899.58 (Δ: -0.61)
  R²: 0.4527 (Δ: +0.0001)
  Size: 1.59 KB (-48.7% smaller)
  Compression: 0.67x


In [6]:
# ============================================================================
# STEP 5: FEATURE SELECTION AFTER QUANTIZATION
# ============================================================================

print("\n" + "="*80)
print("STEP 5: FEATURE SELECTION AFTER QUANTIZATION")
print("="*80)

print("""
WHY FEATURE SELECTION AFTER QUANTIZATION?
- Quantization may change feature importance
- Remove redundant features for further optimization
- Reduce inference time and model complexity
- Improve generalization by reducing overfitting
""")

# ----------------------------------------------------------------------------
# METHOD 1: Correlation-Based Feature Selection
# ----------------------------------------------------------------------------

print("\n" + "-"*80)
print("METHOD 1: CORRELATION-BASED FEATURE SELECTION")
print("-"*80)

def correlation_feature_selection(X, y, threshold=0.1):
    """
    Select features based on correlation with target.
    """
    correlations = X.corrwith(y).abs().sort_values(ascending=False)
    
    print("\nFeature correlations with target:")
    for feat, corr in correlations.items():
        print(f"  {feat}: {corr:.4f}")
    
    selected_features = correlations[correlations > threshold].index.tolist()
    
    return selected_features, correlations

corr_features, correlations = correlation_feature_selection(X_train, y_train, threshold=0.1)

print(f"\n✓ Selected {len(corr_features)} features with correlation > 0.1")
print(f"  Features: {corr_features}")

# ----------------------------------------------------------------------------
# METHOD 2: SelectKBest (Statistical Test)
# ----------------------------------------------------------------------------

print("\n" + "-"*80)
print("METHOD 2: SELECTKBEST (F-STATISTIC)")
print("-"*80)

def selectkbest_feature_selection(X_train, y_train, k=5):
    """
    Select K best features using F-statistic.
    """
    selector = SelectKBest(score_func=f_regression, k=k)
    selector.fit(X_train, y_train)
    
    # Get scores
    scores = pd.DataFrame({
        'feature': X_train.columns,
        'score': selector.scores_
    }).sort_values('score', ascending=False)
    
    print(f"\nFeature scores (F-statistic):")
    print(scores.to_string(index=False))
    
    selected_features = X_train.columns[selector.get_support()].tolist()
    
    return selected_features, scores

kbest_features, kbest_scores = selectkbest_feature_selection(X_train, y_train, k=5)

print(f"\n✓ Selected top {len(kbest_features)} features")
print(f"  Features: {kbest_features}")

# ----------------------------------------------------------------------------
# METHOD 3: Mutual Information
# ----------------------------------------------------------------------------

print("\n" + "-"*80)
print("METHOD 3: MUTUAL INFORMATION")
print("-"*80)

def mutual_info_feature_selection(X_train, y_train, k=5):
    """
    Select features using mutual information.
    """
    mi_scores = mutual_info_regression(X_train, y_train, random_state=42)
    
    mi_df = pd.DataFrame({
        'feature': X_train.columns,
        'mi_score': mi_scores
    }).sort_values('mi_score', ascending=False)
    
    print(f"\nMutual Information scores:")
    print(mi_df.to_string(index=False))
    
    selected_features = mi_df.head(k)['feature'].tolist()
    
    return selected_features, mi_df

mi_features, mi_scores = mutual_info_feature_selection(X_train, y_train, k=5)

print(f"\n✓ Selected top {len(mi_features)} features")
print(f"  Features: {mi_features}")

# ----------------------------------------------------------------------------
# METHOD 4: Recursive Feature Elimination (RFE)
# ----------------------------------------------------------------------------

print("\n" + "-"*80)
print("METHOD 4: RECURSIVE FEATURE ELIMINATION (RFE)")
print("-"*80)

def rfe_feature_selection(X_train, y_train, n_features=5):
    """
    Select features using Recursive Feature Elimination.
    """
    estimator = LinearRegression()
    selector = RFE(estimator, n_features_to_select=n_features, step=1)
    selector.fit(X_train, y_train)
    
    # Get ranking
    ranking = pd.DataFrame({
        'feature': X_train.columns,
        'ranking': selector.ranking_,
        'selected': selector.support_
    }).sort_values('ranking')
    
    print(f"\nRFE Feature Ranking:")
    print(ranking.to_string(index=False))
    
    selected_features = X_train.columns[selector.support_].tolist()
    
    return selected_features, ranking

rfe_features, rfe_ranking = rfe_feature_selection(X_train, y_train, n_features=5)

print(f"\n✓ Selected {len(rfe_features)} features using RFE")
print(f"  Features: {rfe_features}")

# ----------------------------------------------------------------------------
# METHOD 5: Forward Selection
# ----------------------------------------------------------------------------

print("\n" + "-"*80)
print("METHOD 5: FORWARD FEATURE SELECTION")
print("-"*80)

def forward_feature_selection(X_train, y_train, n_features=5):
    """
    Select features using forward selection.
    """
    estimator = LinearRegression()
    selector = SequentialFeatureSelector(
        estimator, 
        n_features_to_select=n_features,
        direction='forward',
        cv=3,
        n_jobs=-1
    )
    
    print(f"Running forward selection (this may take a moment)...")
    selector.fit(X_train, y_train)
    
    selected_features = X_train.columns[selector.get_support()].tolist()
    
    return selected_features

forward_features = forward_feature_selection(X_train, y_train, n_features=5)

print(f"\n✓ Selected {len(forward_features)} features")
print(f"  Features: {forward_features}")

# ----------------------------------------------------------------------------
# COMBINE ALL METHODS: ENSEMBLE FEATURE SELECTION
# ----------------------------------------------------------------------------

print("\n" + "="*80)
print("ENSEMBLE FEATURE SELECTION")
print("="*80)

# Count how many times each feature was selected
all_selections = {
    'Correlation': corr_features,
    'SelectKBest': kbest_features,
    'Mutual Info': mi_features,
    'RFE': rfe_features,
    'Forward': forward_features
}

feature_votes = {}
for method, features in all_selections.items():
    for feature in features:
        if feature not in feature_votes:
            feature_votes[feature] = {'count': 0, 'methods': []}
        feature_votes[feature]['count'] += 1
        feature_votes[feature]['methods'].append(method)

# Sort by vote count
feature_votes_df = pd.DataFrame([
    {
        'feature': feat,
        'votes': info['count'],
        'methods': ', '.join(info['methods'])
    }
    for feat, info in feature_votes.items()
]).sort_values('votes', ascending=False)

print("\nFeature Selection Voting:")
print(feature_votes_df.to_string(index=False))

# Select features with majority vote (3 or more methods)
ensemble_features = feature_votes_df[feature_votes_df['votes'] >= 3]['feature'].tolist()

print(f"\n✓ Ensemble selected features (voted by ≥3 methods): {len(ensemble_features)}")
print(f"  Features: {ensemble_features}")

# If too few features, take top 5 by votes
if len(ensemble_features) < 3:
    ensemble_features = feature_votes_df.head(5)['feature'].tolist()
    print(f"\n  Adjusted to top 5 features: {ensemble_features}")

# ----------------------------------------------------------------------------
# EVALUATE MODELS WITH DIFFERENT FEATURE SETS
# ----------------------------------------------------------------------------

print("\n" + "="*80)
print("EVALUATING MODELS WITH SELECTED FEATURES")
print("="*80)

# Add ensemble BEFORE iteration to avoid changing dict during iteration
all_selections['Ensemble'] = ensemble_features

feature_selection_results = []

# Now iterate over the complete dictionary
for method_name, selected_features in all_selections.items():
    # Train new model with selected features
    X_train_selected = X_train[selected_features]
    X_test_selected = X_test[selected_features]
    
    # Train model
    model = LinearRegression()
    model.fit(X_train_selected, y_train)
    
    # Apply quantization
    quantized = QuantizedLinearRegression(model, dtype='int8')
    quantized.feature_names_ = selected_features
    
    # Evaluate
    y_pred = quantized.predict(X_test_selected)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_selected, y_train, cv=5, scoring='r2')
    
    # Save model
    model_file = f'quantized_model_{method_name.lower().replace(" ", "_")}.npz'
    quantized.save(model_file)
    model_size = os.path.getsize(model_file) / 1024
    
    result = {
        'method': method_name,
        'n_features': len(selected_features),
        'features': selected_features,
        'mse': mse,
        'mae': mae,
        'r2': r2,
        'cv_r2_mean': cv_scores.mean(),
        'cv_r2_std': cv_scores.std(),
        'model_size_kb': model_size,
        'model': quantized,
        'X_test_selected': X_test_selected
    }
    
    feature_selection_results.append(result)
    
    print(f"\n{method_name}:")
    print(f"  Features ({len(selected_features)}): {selected_features}")
    print(f"  MSE: {mse:.2f}")
    print(f"  MAE: {mae:.2f}")
    print(f"  R²: {r2:.4f}")
    print(f"  CV R² (mean±std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"  Model size: {model_size:.2f} KB")

# Select best model
best_model_result = max(feature_selection_results, key=lambda x: x['r2'])

print("\n" + "="*80)
print("BEST MODEL SELECTED")
print("="*80)
print(f"Method: {best_model_result['method']}")
print(f"Features ({best_model_result['n_features']}): {best_model_result['features']}")
print(f"R²: {best_model_result['r2']:.4f}")
print(f"MSE: {best_model_result['mse']:.2f}")
print(f"Size: {best_model_result['model_size_kb']:.2f} KB")


STEP 5: FEATURE SELECTION AFTER QUANTIZATION

WHY FEATURE SELECTION AFTER QUANTIZATION?
- Quantization may change feature importance
- Remove redundant features for further optimization
- Reduce inference time and model complexity
- Improve generalization by reducing overfitting


--------------------------------------------------------------------------------
METHOD 1: CORRELATION-BASED FEATURE SELECTION
--------------------------------------------------------------------------------

Feature correlations with target:
  bmi: 0.6048
  s5: 0.5522
  bp: 0.4448
  s4: 0.4251
  s6: 0.3904
  s3: 0.3840
  s1: 0.1995
  age: 0.1965
  s2: 0.1549
  sex: 0.0071

✓ Selected 9 features with correlation > 0.1
  Features: ['bmi', 's5', 'bp', 's4', 's6', 's3', 's1', 'age', 's2']

--------------------------------------------------------------------------------
METHOD 2: SELECTKBEST (F-STATISTIC)
--------------------------------------------------------------------------------

Feature scores (F-statisti

In [7]:
# ============================================================================
# STEP 6: STRESS TESTING
# ============================================================================

print("\n" + "="*80)
print("STEP 6: COMPREHENSIVE STRESS TESTING")
print("="*80)

print("""
STRESS TEST SCENARIOS:
1. High Volume Load Test (10,000+ requests)
2. Concurrent Request Handling (multi-threading)
3. Memory Leak Detection (continuous operation)
4. Latency Under Load (response time degradation)
5. Edge Cases & Boundary Conditions
6. Long-Running Stability Test
7. Resource Utilization Monitoring
""")

best_model = best_model_result['model']
X_test_best = best_model_result['X_test_selected']

# ----------------------------------------------------------------------------
# STRESS TEST 1: HIGH VOLUME LOAD TEST
# ----------------------------------------------------------------------------

print("\n" + "-"*80)
print("STRESS TEST 1: HIGH VOLUME LOAD TEST")
print("-"*80)

def high_volume_load_test(model, X_data, num_requests=10000):
    """
    Test model with high volume of requests.
    """
    print(f"Running {num_requests} sequential predictions...")
    
    times = []
    predictions = []
    memory_usage = []
    
    process = psutil.Process()
    start_memory = process.memory_info().rss / 1024 / 1024  # MB
    
    overall_start = time.perf_counter()
    
    for i in range(num_requests):
        sample = X_data.iloc[i % len(X_data):i % len(X_data) + 1]
        
        pred_start = time.perf_counter()
        pred = model.predict(sample)
        pred_end = time.perf_counter()
        
        times.append((pred_end - pred_start) * 1000)
        predictions.append(pred[0])
        
        # Memory check every 1000 requests
        if (i + 1) % 1000 == 0:
            current_memory = process.memory_info().rss / 1024 / 1024
            memory_usage.append(current_memory - start_memory)
            print(f"  Progress: {i+1}/{num_requests} | Avg: {np.mean(times[-1000:]):.4f} ms | Memory: +{memory_usage[-1]:.2f} MB")
    
    overall_end = time.perf_counter()
    total_time = overall_end - overall_start
    
    return {
        'num_requests': num_requests,
        'total_time_sec': total_time,
        'avg_time_ms': np.mean(times),
        'median_time_ms': np.median(times),
        'min_time_ms': np.min(times),
        'max_time_ms': np.max(times),
        'p95_time_ms': np.percentile(times, 95),
        'p99_time_ms': np.percentile(times, 99),
        'p999_time_ms': np.percentile(times, 99.9),
        'std_time_ms': np.std(times),
        'throughput_per_sec': num_requests / total_time,
        'times': times,
        'memory_increase_mb': memory_usage[-1] if memory_usage else 0,
        'memory_trend': memory_usage
    }

load_test_result = high_volume_load_test(best_model, X_test_best, num_requests=10000)

print(f"\n✓ Load Test Complete")
print(f"  Total requests: {load_test_result['num_requests']:,}")
print(f"  Total time: {load_test_result['total_time_sec']:.2f} seconds")
print(f"  Avg response time: {load_test_result['avg_time_ms']:.4f} ms")
print(f"  Median response time: {load_test_result['median_time_ms']:.4f} ms")
print(f"  P95 latency: {load_test_result['p95_time_ms']:.4f} ms")
print(f"  P99 latency: {load_test_result['p99_time_ms']:.4f} ms")
print(f"  P99.9 latency: {load_test_result['p999_time_ms']:.4f} ms")
print(f"  Throughput: {load_test_result['throughput_per_sec']:.2f} req/sec")
print(f"  Memory increase: +{load_test_result['memory_increase_mb']:.2f} MB")

# ----------------------------------------------------------------------------
# STRESS TEST 2: CONCURRENT REQUEST HANDLING
# ----------------------------------------------------------------------------

print("\n" + "-"*80)
print("STRESS TEST 2: CONCURRENT REQUEST HANDLING")
print("-"*80)

def concurrent_stress_test(model, X_data, num_requests=1000, num_workers=10):
    """
    Test model with concurrent requests.
    """
    print(f"Running {num_requests} requests with {num_workers} concurrent workers...")
    
    def predict_one(idx):
        sample = X_data.iloc[idx % len(X_data):idx % len(X_data) + 1]
        start = time.perf_counter()
        pred = model.predict(sample)
        end = time.perf_counter()
        return (end - start) * 1000, pred[0]
    
    start_time = time.perf_counter()
    
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        results = list(executor.map(predict_one, range(num_requests)))
    
    end_time = time.perf_counter()
    
    times = [r[0] for r in results]
    predictions = [r[1] for r in results]
    
    return {
        'num_requests': num_requests,
        'num_workers': num_workers,
        'total_time_sec': end_time - start_time,
        'avg_time_ms': np.mean(times),
        'p95_time_ms': np.percentile(times, 95),
        'p99_time_ms': np.percentile(times, 99),
        'throughput_per_sec': num_requests / (end_time - start_time),
        'times': times
    }

concurrent_results = []
worker_counts = [1, 2, 4, 8, 16]

for workers in worker_counts:
    result = concurrent_stress_test(best_model, X_test_best, num_requests=1000, num_workers=workers)
    concurrent_results.append(result)
    
    print(f"\n  Workers: {workers}")
    print(f"    Total time: {result['total_time_sec']:.2f} sec")
    print(f"    Avg response: {result['avg_time_ms']:.4f} ms")
    print(f"    P95: {result['p95_time_ms']:.4f} ms")
    print(f"    Throughput: {result['throughput_per_sec']:.2f} req/sec")

# ----------------------------------------------------------------------------
# STRESS TEST 3: MEMORY LEAK DETECTION
# ----------------------------------------------------------------------------

print("\n" + "-"*80)
print("STRESS TEST 3: MEMORY LEAK DETECTION")
print("-"*80)

def memory_leak_test(model, X_data, duration_seconds=10, check_interval=1):
    """
    Monitor memory usage over time to detect leaks.
    """
    print(f"Running continuous predictions for {duration_seconds} seconds...")
    
    process = psutil.Process()
    start_memory = process.memory_info().rss / 1024 / 1024
    
    memory_samples = []
    prediction_count = 0
    
    start_time = time.time()
    last_check = start_time
    
    while (time.time() - start_time) < duration_seconds:
        # Make predictions
        sample = X_data.iloc[prediction_count % len(X_data):prediction_count % len(X_data) + 1]
        pred = model.predict(sample)
        prediction_count += 1
        
        # Check memory periodically
        if (time.time() - last_check) >= check_interval:
            current_memory = process.memory_info().rss / 1024 / 1024
            memory_samples.append({
                'time': time.time() - start_time,
                'memory_mb': current_memory,
                'memory_delta_mb': current_memory - start_memory,
                'predictions_made': prediction_count
            })
            last_check = time.time()
    
    end_memory = process.memory_info().rss / 1024 / 1024
    
    # Detect memory leak (simple linear regression on memory over time)
    if len(memory_samples) > 2:
        times = np.array([s['time'] for s in memory_samples])
        mems = np.array([s['memory_delta_mb'] for s in memory_samples])
        
        # Calculate slope (memory increase per second)
        slope = np.polyfit(times, mems, 1)[0]
        leak_detected = slope > 0.1  # More than 0.1 MB/sec increase
    else:
        slope = 0
        leak_detected = False
    
    return {
        'duration_sec': duration_seconds,
        'predictions_made': prediction_count,
        'start_memory_mb': start_memory,
        'end_memory_mb': end_memory,
        'memory_increase_mb': end_memory - start_memory,
        'memory_samples': memory_samples,
        'memory_leak_rate_mb_per_sec': slope,
        'leak_detected': leak_detected
    }

memory_result = memory_leak_test(best_model, X_test_best, duration_seconds=10)

print(f"\n✓ Memory Leak Test Complete")
print(f"  Duration: {memory_result['duration_sec']} seconds")
print(f"  Predictions made: {memory_result['predictions_made']:,}")
print(f"  Start memory: {memory_result['start_memory_mb']:.2f} MB")
print(f"  End memory: {memory_result['end_memory_mb']:.2f} MB")
print(f"  Memory increase: {memory_result['memory_increase_mb']:.2f} MB")
print(f"  Leak rate: {memory_result['memory_leak_rate_mb_per_sec']:.4f} MB/sec")
print(f"  Leak detected: {'⚠️ YES' if memory_result['leak_detected'] else '✓ NO'}")

# ----------------------------------------------------------------------------
# STRESS TEST 4: LATENCY UNDER INCREASING LOAD
# ----------------------------------------------------------------------------

print("\n" + "-"*80)
print("STRESS TEST 4: LATENCY UNDER INCREASING LOAD")
print("-"*80)

def latency_under_load_test(model, X_data):
    """
    Test how latency changes with increasing load.
    """
    load_levels = [10, 50, 100, 500, 1000, 5000, 10000]
    results = []
    
    for load in load_levels:
        times = []
        
        for i in range(load):
            sample = X_data.iloc[i % len(X_data):i % len(X_data) + 1]
            
            start = time.perf_counter()
            pred = model.predict(sample)
            end = time.perf_counter()
            
            times.append((end - start) * 1000)
        
        results.append({
            'load': load,
            'avg_latency_ms': np.mean(times),
            'p95_latency_ms': np.percentile(times, 95),
            'p99_latency_ms': np.percentile(times, 99)
        })
        
        print(f"  Load: {load:>6} | Avg: {np.mean(times):.4f} ms | P95: {np.percentile(times, 95):.4f} ms")
    
    return results

print("Testing latency under increasing load...")
latency_results = latency_under_load_test(best_model, X_test_best)

# Check for latency degradation
first_avg = latency_results[0]['avg_latency_ms']
last_avg = latency_results[-1]['avg_latency_ms']
degradation_pct = ((last_avg - first_avg) / first_avg) * 100

print(f"\n✓ Latency degradation: {degradation_pct:+.2f}%")
print(f"  {'✓ Stable' if abs(degradation_pct) < 20 else '⚠️ Significant degradation'}")

# ----------------------------------------------------------------------------
# STRESS TEST 5: EDGE CASES & BOUNDARY CONDITIONS
# ----------------------------------------------------------------------------

print("\n" + "-"*80)
print("STRESS TEST 5: EDGE CASES & BOUNDARY CONDITIONS")
print("-"*80)

def edge_case_test(model, X_data):
    """
    Test model with edge cases and boundary values.
    """
    test_cases = []
    
    # Test 1: Zero values
    zero_input = pd.DataFrame(
        np.zeros((1, len(X_data.columns))),
        columns=X_data.columns
    )
    try:
        pred = model.predict(zero_input)
        test_cases.append({'case': 'All zeros', 'status': 'PASS', 'prediction': pred[0]})
    except Exception as e:
        test_cases.append({'case': 'All zeros', 'status': 'FAIL', 'error': str(e)})
    
    # Test 2: Maximum values
    max_input = pd.DataFrame(
        X_data.max().values.reshape(1, -1),
        columns=X_data.columns
    )
    try:
        pred = model.predict(max_input)
        test_cases.append({'case': 'Max values', 'status': 'PASS', 'prediction': pred[0]})
    except Exception as e:
        test_cases.append({'case': 'Max values', 'status': 'FAIL', 'error': str(e)})
    
    # Test 3: Minimum values
    min_input = pd.DataFrame(
        X_data.min().values.reshape(1, -1),
        columns=X_data.columns
    )
    try:
        pred = model.predict(min_input)
        test_cases.append({'case': 'Min values', 'status': 'PASS', 'prediction': pred[0]})
    except Exception as e:
        test_cases.append({'case': 'Min values', 'status': 'FAIL', 'error': str(e)})
    
    # Test 4: Negative values
    negative_input = pd.DataFrame(
        -np.abs(X_data.iloc[0].values.reshape(1, -1)),
        columns=X_data.columns
    )
    try:
        pred = model.predict(negative_input)
        test_cases.append({'case': 'Negative values', 'status': 'PASS', 'prediction': pred[0]})
    except Exception as e:
        test_cases.append({'case': 'Negative values', 'status': 'FAIL', 'error': str(e)})
    
    # Test 5: Very large values
    large_input = pd.DataFrame(
        X_data.iloc[0].values.reshape(1, -1) * 1000,
        columns=X_data.columns
    )
    try:
        pred = model.predict(large_input)
        test_cases.append({'case': 'Very large values', 'status': 'PASS', 'prediction': pred[0]})
    except Exception as e:
        test_cases.append({'case': 'Very large values', 'status': 'FAIL', 'error': str(e)})
    
    return test_cases

edge_cases = edge_case_test(best_model, X_test_best)

print("\nEdge Case Test Results:")
for case in edge_cases:
    status_symbol = '✓' if case['status'] == 'PASS' else '✗'
    print(f"  {status_symbol} {case['case']}: {case['status']}")
    if 'prediction' in case:
        print(f"      Prediction: {case['prediction']:.2f}")
    if 'error' in case:
        print(f"      Error: {case['error']}")

# ----------------------------------------------------------------------------
# STRESS TEST 6: LONG-RUNNING STABILITY
# ----------------------------------------------------------------------------

print("\n" + "-"*80)
print("STRESS TEST 6: LONG-RUNNING STABILITY TEST")
print("-"*80)

def stability_test(model, X_data, num_batches=20, batch_size=500):
    """
    Test model stability over extended operation.
    """
    print(f"Running {num_batches} batches of {batch_size} predictions each...")
    
    batch_results = []
    
    for batch_num in range(num_batches):
        batch_times = []
        
        for i in range(batch_size):
            sample = X_data.iloc[i % len(X_data):i % len(X_data) + 1]
            
            start = time.perf_counter()
            pred = model.predict(sample)
            end = time.perf_counter()
            
            batch_times.append((end - start) * 1000)
        
        batch_results.append({
            'batch': batch_num + 1,
            'avg_time_ms': np.mean(batch_times),
            'p95_time_ms': np.percentile(batch_times, 95)
        })
        
        if (batch_num + 1) % 5 == 0:
            print(f"  Batch {batch_num + 1}/{num_batches}: Avg {np.mean(batch_times):.4f} ms")
    
    # Check for performance degradation over time
    first_5_avg = np.mean([b['avg_time_ms'] for b in batch_results[:5]])
    last_5_avg = np.mean([b['avg_time_ms'] for b in batch_results[-5:]])
    degradation = ((last_5_avg - first_5_avg) / first_5_avg) * 100
    
    return {
        'num_batches': num_batches,
        'batch_size': batch_size,
        'total_predictions': num_batches * batch_size,
        'batch_results': batch_results,
        'performance_degradation_pct': degradation,
        'stable': abs(degradation) < 10
    }

stability_result = stability_test(best_model, X_test_best, num_batches=20, batch_size=500)

print(f"\n✓ Stability Test Complete")
print(f"  Total predictions: {stability_result['total_predictions']:,}")
print(f"  Performance degradation: {stability_result['performance_degradation_pct']:+.2f}%")
print(f"  Stability: {'✓ STABLE' if stability_result['stable'] else '⚠️ UNSTABLE'}")

# ----------------------------------------------------------------------------
# STRESS TEST 7: RESOURCE UTILIZATION MONITORING
# ----------------------------------------------------------------------------

print("\n" + "-"*80)
print("STRESS TEST 7: RESOURCE UTILIZATION MONITORING")
print("-"*80)

def resource_monitoring_test(model, X_data, num_predictions=5000):
    """
    Monitor CPU and memory usage during predictions.
    """
    print(f"Monitoring resources during {num_predictions} predictions...")
    
    process = psutil.Process()
    
    cpu_samples = []
    memory_samples = []
    
    for i in range(num_predictions):
        sample = X_data.iloc[i % len(X_data):i % len(X_data) + 1]
        
        # Measure before prediction
        cpu_before = process.cpu_percent()
        mem_before = process.memory_info().rss / 1024 / 1024
        
        pred = model.predict(sample)
        
        # Measure after prediction
        cpu_after = process.cpu_percent()
        mem_after = process.memory_info().rss / 1024 / 1024
        
        if i % 500 == 0:
            cpu_samples.append(cpu_after)
            memory_samples.append(mem_after)
    
    return {
        'avg_cpu_percent': np.mean(cpu_samples),
        'max_cpu_percent': np.max(cpu_samples),
        'avg_memory_mb': np.mean(memory_samples),
        'max_memory_mb': np.max(memory_samples),
        'cpu_samples': cpu_samples,
        'memory_samples': memory_samples
    }

resource_result = resource_monitoring_test(best_model, X_test_best, num_predictions=5000)

print(f"\n✓ Resource Monitoring Complete")
print(f"  Avg CPU usage: {resource_result['avg_cpu_percent']:.2f}%")
print(f"  Max CPU usage: {resource_result['max_cpu_percent']:.2f}%")
print(f"  Avg Memory: {resource_result['avg_memory_mb']:.2f} MB")
print(f"  Max Memory: {resource_result['max_memory_mb']:.2f} MB")


STEP 6: COMPREHENSIVE STRESS TESTING

STRESS TEST SCENARIOS:
1. High Volume Load Test (10,000+ requests)
2. Concurrent Request Handling (multi-threading)
3. Memory Leak Detection (continuous operation)
4. Latency Under Load (response time degradation)
5. Edge Cases & Boundary Conditions
6. Long-Running Stability Test
7. Resource Utilization Monitoring


--------------------------------------------------------------------------------
STRESS TEST 1: HIGH VOLUME LOAD TEST
--------------------------------------------------------------------------------
Running 10000 sequential predictions...
  Progress: 1000/10000 | Avg: 0.0274 ms | Memory: +0.00 MB
  Progress: 2000/10000 | Avg: 0.0328 ms | Memory: +0.12 MB
  Progress: 3000/10000 | Avg: 0.0175 ms | Memory: +0.25 MB
  Progress: 4000/10000 | Avg: 0.0155 ms | Memory: +0.25 MB
  Progress: 5000/10000 | Avg: 0.0155 ms | Memory: +0.38 MB
  Progress: 6000/10000 | Avg: 0.0183 ms | Memory: +0.38 MB
  Progress: 7000/10000 | Avg: 0.0158 ms | Memory: 